# 04 Portfolio Walk-Forward

This notebook checks portfolio OOS stability across multiple time windows.

Key questions:

- How many windows are positive?
- How bad is the worst window?
- Is OOS drawdown stable?
- Do Sharpe/PF hold across many regimes or only a few windows?


In [ ]:
#
#

import sys
from pathlib import Path


def _find_root(start: Path, marker: str = 'pyproject.toml') -> Path:
    for p in [start, *start.parents]:
        if (p / marker).exists() and (p / 'core_python' / 'shared').exists():
            return p
    raise RuntimeError(f'Could not find repo root containing {marker!r} and core_python/shared')


ROOT = _find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)
print('CORE =', CORE)


In [ ]:

from IPython.display import display

from core_python.strategies.combo.params import summary as strategy_summary
from core_python.strategies.combo.research_utils import (
    configure_notebook,
    plot_walkforward_dashboard,
    show_note,
    show_run_config,
    summarize_walkforward,
)
from core_python.strategies.combo.portfolio.walkforward import walk_forward_portfolio

configure_notebook()
print(strategy_summary())


In [ ]:
#
# IS_BARS: so bar dung truoc OOS de dinh vi cua so; file nay khong re-optimize.

SYMBOL_PARAMS = {
    'US30':  {'x': 10.0, 'ktp': 2.3, 'ma_period': 20, 'trailing_activation': 1.0},
    'US500': {'x': 1.0,  'ktp': 2.3, 'ma_period': 20, 'trailing_activation': 1.0},
    'DE40':  {'x': 5.0,  'ktp': 2.3, 'ma_period': 20, 'trailing_activation': 1.0},
    'GOLD':  {'x': 0.5,  'ktp': 2.3, 'ma_period': 20, 'trailing_activation': 1.0},
}

RUN_CONFIG = {
    'account_mode': 'standard',
    'initial_balance': 100_000.0,
    'is_bars': 5_000,
    'oos_bars': 1_250,
    'step_bars': 1_250,
    'max_bars': 40_000,
}

show_run_config('Portfolio Walk-Forward Configuration', RUN_CONFIG)
show_note('Symbol Params', 'These parameters are being evaluated across OOS windows.')
display(SYMBOL_PARAMS)


In [ ]:

wf_df, wf_summary = walk_forward_portfolio(
    SYMBOL_PARAMS,
    account_mode=RUN_CONFIG['account_mode'],
    initial_balance=RUN_CONFIG['initial_balance'],
    is_bars=RUN_CONFIG['is_bars'],
    oos_bars=RUN_CONFIG['oos_bars'],
    step_bars=RUN_CONFIG['step_bars'],
    max_bars=RUN_CONFIG['max_bars'],
)

show_note('Raw Walk-Forward Summary', 'Raw runner summary kept for cross-checking.')
print(wf_summary)
display(wf_df)


In [ ]:
# Cell 5 - Stability dashboard
#

wf_stability = summarize_walkforward(wf_df)
plot_walkforward_dashboard(wf_df)

metric_cols = [
    c for c in ['window', 'oos_start', 'oos_end', 'total_return', 'max_drawdown', 'sharpe', 'profit_factor', 'win_rate']
    if c in wf_df.columns
]
if metric_cols:
    display(wf_df[metric_cols])


In [ ]:
#

EXPORT_REPORT = False
if EXPORT_REPORT and not wf_df.empty:
    out_dir = ROOT / 'reports' / 'combo' / 'portfolio_walkforward'
    out_dir.mkdir(parents=True, exist_ok=True)
    wf_df.to_csv(out_dir / 'walkforward_windows.csv', index=False, encoding='utf-8-sig')
    wf_stability.to_csv(out_dir / 'walkforward_summary.csv', encoding='utf-8-sig')
    print('Exported to:', out_dir)
else:
    print('Export is disabled. Set EXPORT_REPORT = True to save CSV files.')
